# Adaptive tree: Plummer sphere, interactions and static execution

This notebook separates tree construction from numerical-plan construction, then compares adaptive and uniform trees for the same reproducible truncated Plummer-sphere particle distribution. Adaptive and uniform maximum depths are configured independently. Every dipole has the same magnitude and an independently sampled isotropic orientation.

Geometry and preflight cells work without CUDA. Numerical plans are built only after the topology inspection, and an adjustable memory guard prevents accidentally materialising an unsafe number of near-field particle pairs. The benchmark cells use the P2P packing and executor selected in the configuration cell.

In [ ]:
import time
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
import cdfmm
try:
    from adaptive_showcase import *
except ModuleNotFoundError:
    from examples.notebooks.adaptive_showcase import *

N = int(2**15)
PLUMMER_A = 1.0
PLUMMER_R_MAX = 5.0 * PLUMMER_A
MOMENT_MAGNITUDE = 1.0
SEED = 42

LEAF_CAPACITY = 32
# These depths are independent. Different values compare different accuracy/work trade-offs.
ADAPTIVE_MAX_DEPTH = 7
UNIFORM_MAX_DEPTH = 5
BACKEND = cdfmm.ExecutionBackend.CUDA_FULL
PRECISION = cdfmm.StaticPrecision.FLOAT32
ORDER = 6
N_EVALUATIONS = 10

RUN_FULL_SWEEP = False
BENCHMARK_INTERACTION_MODES = (INTERACTION_MODES if RUN_FULL_SWEEP
                               else ("point-point",))
# Select one mode for a quick run, or enable the sweep for all P2P paths.
#P2P_MODE = "bsr"  # canonical, bsr, dictionary
P2P_MODE = "dictionary"  # canonical, bsr, dictionary
DICTIONARY_EXECUTOR = "target_owned"  # source_warp, target_owned, power2
# With P2P_MODE == "bsr", optionally add exactly one reduced comparison.
RUN_BSR_COMPARISON = False
BSR_COMPARISON_REDUCED_KERNEL = "power2"  # warp, target, power2
RUN_P2P_SWEEP = False
_BSR_REDUCED_EXECUTORS = {
    "warp": "source_warp",
    "target": "target_owned",
    "power2": "power2",
}
if BSR_COMPARISON_REDUCED_KERNEL not in _BSR_REDUCED_EXECUTORS:
    raise ValueError("BSR_COMPARISON_REDUCED_KERNEL must be warp, target, or power2")
if RUN_P2P_SWEEP:
    BENCHMARK_P2P_MODES = P2P_MODES
    BENCHMARK_DICTIONARY_EXECUTORS = DICTIONARY_EXECUTORS
elif RUN_BSR_COMPARISON:
    if P2P_MODE != "bsr":
        raise ValueError("RUN_BSR_COMPARISON requires P2P_MODE = 'bsr'")
    BENCHMARK_P2P_MODES = ("bsr", "dictionary")
    BENCHMARK_DICTIONARY_EXECUTORS = (
        _BSR_REDUCED_EXECUTORS[BSR_COMPARISON_REDUCED_KERNEL],
    )
else:
    BENCHMARK_P2P_MODES = (P2P_MODE,)
    BENCHMARK_DICTIONARY_EXECUTORS = (DICTIONARY_EXECUTOR,)
REDUCED_VALUES = tuple(mode == "dictionary" for mode in BENCHMARK_P2P_MODES)

# Current plan construction temporarily retains several P2P representations.
# The estimate is approximate; the stop can be overridden deliberately.
PLAN_BUILD_WARNING_GIB = 8.0
PLAN_BUILD_STOP_GIB = 64.0
ALLOW_LARGE_PLAN_BUILD = False

REFERENCE_TARGET_COUNT = 256
REFERENCE_RUNS = (0, -1)
SELECTED_LEAF = None
INCLUDE_ANCESTORS = True
RUN_BENCHMARK = True

## 1. Generate the truncated Plummer sphere

The notebook draws exactly `N` positions from a spherical Plummer profile with scale radius $a=1$, truncated at $r_{\max}=10a$. Radii use the analytic inverse CDF and directions are isotropic. Every point receives an independent isotropic dipole orientation with the same magnitude `MOMENT_MAGNITUDE`. A fixed seed makes both positions and moments reproducible.

The empirical radial CDF is plotted against the analytic truncated CDF as a sampling check. This configuration evaluates point dipoles only; both tree builders and every benchmark case receive the same position and moment arrays.

In [ ]:
def generate_plummer_particles(n, seed, a=1.0, r_max=10.0, m0=1.0):
    if n < 1:
        raise ValueError("N must be positive")
    if a <= 0.0 or r_max <= 0.0 or m0 <= 0.0:
        raise ValueError("a, r_max, and m0 must be positive")
    rng = np.random.default_rng(seed)
    f_max = r_max**3 / (r_max**2 + a**2)**1.5
    q = rng.random(n) * f_max
    with np.errstate(divide="ignore"):
        radii = a / np.sqrt(q**(-2.0/3.0) - 1.0)

    position_directions = rng.normal(size=(n, 3))
    position_directions /= np.linalg.norm(position_directions, axis=1)[:, None]
    positions = radii[:, None] * position_directions

    moment_directions = rng.normal(size=(n, 3))
    moment_directions /= np.linalg.norm(moment_directions, axis=1)[:, None]
    moments = m0 * moment_directions
    np.testing.assert_allclose(np.linalg.norm(moments, axis=1), m0, rtol=1e-14)
    if len(positions) != n or np.max(radii) > r_max:
        raise RuntimeError("invalid truncated Plummer sample")
    return dict(
        positions=np.ascontiguousarray(positions),
        moments=np.ascontiguousarray(moments),
        magnetisations=np.ascontiguousarray(moment_directions),
        radii=radii,
        # Point mode does not consume sizes, but the shared option helper expects the key.
        sides=np.zeros(n, dtype=float),
    )

material = generate_plummer_particles(
    N, SEED, PLUMMER_A, PLUMMER_R_MAX, MOMENT_MAGNITUDE)
positions = material["positions"]
radii = material["radii"]
print(f"Geometry: truncated Plummer sphere; particles: {len(positions):,}")
print(f"a = {PLUMMER_A:g}; r_max = {PLUMMER_R_MAX:g}; "
      f"sampled radius range = [{radii.min():.4g}, {radii.max():.4g}]")
print("dipole magnitude range:",
      np.linalg.norm(material["moments"], axis=1).min(),
      np.linalg.norm(material["moments"], axis=1).max())

fig = plt.figure(figsize=(13, 5))
axis = fig.add_subplot(121, projection="3d")
sample = np.linspace(0, len(positions) - 1, min(5000, len(positions)), dtype=int)
scatter = axis.scatter(*positions[sample].T, c=radii[sample], s=4,
                       cmap="viridis", alpha=.65)
fig.colorbar(scatter, ax=axis, label="radius", shrink=.6)
arrows = sample[::max(1, len(sample)//30)]
axis.quiver(*positions[arrows].T, *material["moments"][arrows].T,
            length=.35, normalize=True, color="black", linewidth=.5)
finish_3d_axes(axis, "Truncated Plummer particle sample")

axis = fig.add_subplot(122)
sorted_radii = np.sort(radii)
empirical_cdf = np.arange(1, N + 1) / N
radius_grid = np.linspace(0.0, PLUMMER_R_MAX, 500)
f_max = PLUMMER_R_MAX**3 / (PLUMMER_R_MAX**2 + PLUMMER_A**2)**1.5
analytic_cdf = (radius_grid**3 /
                (radius_grid**2 + PLUMMER_A**2)**1.5 / f_max)
axis.plot(sorted_radii, empirical_cdf, label="empirical")
axis.plot(radius_grid, analytic_cdf, "--", label="analytic truncated Plummer")
axis.set(xlabel="radius", ylabel="CDF", title="Radial distribution check",
         xlim=(0.0, PLUMMER_R_MAX), ylim=(0.0, 1.0))
axis.grid(alpha=.25)
axis.legend()
fig.tight_layout()

## 2. Build both trees without numerical operators

Both trees use an origin-centred root cube with half-width `PLUMMER_R_MAX`, so the complete truncated sphere lies inside the same bounds. The uniform tree uses the adaptive tree's reached depth and identical particles. Leaf capacity is a split trigger rather than a guarantee at the depth cap. These objects contain geometry and connectivity only: no expansion basis, translation matrices, coefficient state, backend packing, or GPU upload has been allocated.

In [ ]:
def build_plummer_trees(positions, capacity, adaptive_max_depth,
                        uniform_max_depth, root_half_width):
    adaptive_options = cdfmm.AdaptiveTreeOptions()
    adaptive_options.max_particles_per_leaf = capacity
    adaptive_options.max_depth = adaptive_max_depth
    adaptive_options.root_centre = cdfmm.Vec3(0.0, 0.0, 0.0)
    adaptive_options.root_half_width = root_half_width
    start = time.perf_counter()
    adaptive = cdfmm.AdaptiveTree(positions, adaptive_options)
    adaptive_seconds = time.perf_counter() - start

    uniform_options = cdfmm.UniformTreeOptions()
    uniform_options.max_level = uniform_max_depth
    uniform_options.root_centre = adaptive_options.root_centre
    uniform_options.root_half_width = adaptive_options.root_half_width
    start = time.perf_counter()
    uniform = cdfmm.UniformTree(positions, positions, uniform_options)
    uniform_tree_seconds = time.perf_counter() - start
    start = time.perf_counter()
    uniform_topology = cdfmm.uniform_topology(uniform)
    uniform_topology_seconds = time.perf_counter() - start
    return (
        adaptive, uniform,
        dict(adaptive=adaptive.topology, uniform=uniform_topology),
        dict(
            adaptive=adaptive_seconds,
            uniform=uniform_tree_seconds + uniform_topology_seconds,
            adaptive_tree=adaptive.tree_seconds,
            adaptive_interactions=adaptive.interaction_seconds,
            uniform_tree=uniform_tree_seconds,
            uniform_topology=uniform_topology_seconds,
        ),
    )


adaptive_tree, uniform_tree, topologies, tree_times = build_plummer_trees(
    positions, LEAF_CAPACITY, ADAPTIVE_MAX_DEPTH, UNIFORM_MAX_DEPTH,
    PLUMMER_R_MAX)
tree_order = ["uniform", "adaptive"]
maximum_depths = {
    "adaptive": ADAPTIVE_MAX_DEPTH,
    "uniform": UNIFORM_MAX_DEPTH,
}
capacities = {name: LEAF_CAPACITY for name in tree_order}
summaries = {
    name: summarise(topology, capacities[name], maximum_depths[name])
    for name, topology in topologies.items()
}

topology_table = pd.DataFrame([
    dict(
        Tree=name.title(),
        **{
            "Configured depth": maximum_depths[name],
            "Reached depth": summaries[name]["reached_depth"],
            "Nodes": summaries[name]["nodes"],
            "Occupied leaves": summaries[name]["leaves"],
            "Maximum occupancy": summaries[name]["occupancy_max"],
            "Depth-limited leaves": summaries[name]["depth_limited"],
            "P2P particle pairs": summaries[name]["p2p_pairs"],
            "M2L interactions": summaries[name]["m2l"],
            "Cross-level M2L": summaries[name]["cross_level_m2l"],
            "Topology MiB": summaries[name]["topology_bytes"] / 2**20,
            "Tree + topology ms": tree_times[name] * 1e3,
        },
    )
    for name in tree_order
]).set_index("Tree")
display(topology_table.style.format({
    "Nodes": "{:,.0f}", "Occupied leaves": "{:,.0f}",
    "Maximum occupancy": "{:,.0f}", "Depth-limited leaves": "{:,.0f}",
    "P2P particle pairs": "{:,.0f}", "M2L interactions": "{:,.0f}",
    "Cross-level M2L": "{:,.0f}", "Topology MiB": "{:.1f}",
    "Tree + topology ms": "{:.1f}",
}))

# Estimate scale before Step 4 creates any dense particle-pair operators.
preflight_records = plan_build_preflight(
    topologies, BENCHMARK_P2P_MODES, PRECISION, capacities, maximum_depths)
preflight_table = pd.DataFrame(preflight_records)
preflight_table["status"] = np.where(
    preflight_table.estimated_host_build_peak_gib > PLAN_BUILD_STOP_GIB,
    "BLOCKED",
    np.where(preflight_table.estimated_host_build_peak_gib >
             PLAN_BUILD_WARNING_GIB, "WARNING", "OK"),
)
display(preflight_table.rename(columns={
    "tree": "Tree", "p2p_mode": "P2P mode",
    "configured_max_depth": "Configured depth",
    "reached_depth": "Reached depth", "nodes": "Nodes", "leaves": "Leaves",
    "depth_limited_leaves": "Depth-limited leaves",
    "maximum_leaf_occupancy": "Maximum leaf occupancy",
    "p2p_particle_pairs": "P2P particle pairs",
    "estimated_device_p2p_gib": "Estimated device P2P GiB",
    "estimated_host_build_peak_gib": "Estimated host build peak GiB",
    "status": "Status",
}).style.format({
    "Nodes": "{:,.0f}", "Leaves": "{:,.0f}",
    "Depth-limited leaves": "{:,.0f}", "Maximum leaf occupancy": "{:,.0f}",
    "P2P particle pairs": "{:,.0f}", "Estimated device P2P GiB": "{:.2f}",
    "Estimated host build peak GiB": "{:.1f}",
}))

for name in tree_order:
    summary = summaries[name]
    if summary["depth_limited"]:
        warnings.warn(
            f"{name.title()} has {summary['depth_limited']:,} leaves above "
            f"capacity {LEAF_CAPACITY} at its depth limit; maximum occupancy "
            f"is {summary['occupancy_max']:,}. Near-field work can grow quadratically.",
            RuntimeWarning,
        )
for record in preflight_records:
    if record["estimated_host_build_peak_gib"] > PLAN_BUILD_WARNING_GIB:
        warnings.warn(
            f"{record['tree'].title()} {record['p2p_mode']} plan construction "
            f"may peak near {record['estimated_host_build_peak_gib']:.1f} GiB "
            f"for {record['p2p_particle_pairs']:,} P2P pairs.",
            RuntimeWarning,
        )
if "dictionary" in BENCHMARK_P2P_MODES:
    warnings.warn(
        "Continuous Plummer positions normally give little exact Tensor6 "
        "dictionary reuse beyond reciprocal pairs.", RuntimeWarning)

PLAN_BUILD_BLOCKED = bool(
    (preflight_table.estimated_host_build_peak_gib > PLAN_BUILD_STOP_GIB).any()
    and not ALLOW_LARGE_PLAN_BUILD)
if PLAN_BUILD_BLOCKED:
    display(Markdown(
        f"**Numerical-plan construction is disabled:** at least one estimated "
        f"host peak exceeds `{PLAN_BUILD_STOP_GIB:g} GiB`. Increase tree depth, "
        "reduce `N`, or set `ALLOW_LARGE_PLAN_BUILD = True` deliberately. "
        "The topology and interaction illustrations remain available."))

# Topology workload dashboard. Tree identity is fixed: uniform blue, adaptive red.
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
tree_colours = [TREE_COLOURS[name] for name in tree_order]
x = np.arange(len(tree_order))

width = 0.36
axes[0, 0].bar(x - width/2, [summaries[name]["nodes"] for name in tree_order],
               width, color=tree_colours, alpha=.9, label="all nodes")
axes[0, 0].bar(x + width/2, [summaries[name]["leaves"] for name in tree_order],
               width, color=tree_colours, alpha=.45, hatch="//",
               label="occupied leaves")
axes[0, 0].set(title="Tree size", ylabel="count", yscale="log")
axes[0, 0].legend()

axes[0, 1].bar(x, [summaries[name]["topology_bytes"] / 2**20
                   for name in tree_order], color=tree_colours)
axes[0, 1].set(title="Canonical topology memory", ylabel="MiB")

axes[1, 0].bar(x, [summaries[name]["p2p_pairs"] / 1e6
                   for name in tree_order], color=tree_colours)
axes[1, 0].set(title="Near field: explicit particle pairs", ylabel="million pairs")

same_level = [
    (summaries[name]["m2l"] - summaries[name]["cross_level_m2l"]) / 1e6
    for name in tree_order
]
cross_level = [summaries[name]["cross_level_m2l"] / 1e6
               for name in tree_order]
axes[1, 1].bar(x, same_level, color=tree_colours, label="same level")
axes[1, 1].bar(x, cross_level, bottom=same_level, color=tree_colours,
               alpha=.45, hatch="xx", label="cross level")
axes[1, 1].set(title="Far field: M2L translations", ylabel="million interactions")
axes[1, 1].legend()
for axis in axes.flat:
    axis.set_xticks(x, [name.title() for name in tree_order])
    axis.grid(axis="y", alpha=.25)
fig.suptitle("Topology and interaction workload", fontsize=14)
fig.tight_layout()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
maximum_occupancy = max(summaries[name]["occupancy_max"] for name in tree_order)
bins = np.geomspace(1, max(2, maximum_occupancy + 1), 28)
for name in tree_order:
    topology = topologies[name]
    axes[0].hist([leaf.count for leaf in topology.source_leaves], bins=bins,
                 histtype="step", linewidth=2, color=TREE_COLOURS[name],
                 label=name.title())
    counts = summaries[name]["leaves_by_level"]
    axes[1].plot(list(counts), list(counts.values()), "o-", linewidth=2,
                 color=TREE_COLOURS[name], label=name.title())
axes[0].axvline(LEAF_CAPACITY, color="black", linestyle=":",
                label=f"adaptive capacity ({LEAF_CAPACITY})")
axes[0].set(xlabel="dipoles per occupied leaf", ylabel="leaf count",
            title="Leaf occupancy distribution", xscale="log", yscale="log")
axes[1].set(xlabel="level", ylabel="occupied leaves",
            title="Occupied leaves at each depth")
for axis in axes:
    axis.grid(alpha=.25)
    axis.legend()
fig.tight_layout()

fig = plt.figure(figsize=(12, 5))
for panel, name in enumerate(tree_order, 1):
    topology = topologies[name]
    axis = fig.add_subplot(1, 2, panel, projection="3d")
    colour_depth = max(1, maximum_depths[name])
    for leaf in topology.target_leaves:
        node = topology.nodes[leaf.node]
        draw_box_3d(axis, vec3_to_array(node.centre), node.half_width,
                    colour=plt.cm.viridis(node.level / colour_depth),
                    alpha=.5, linewidth=.5)
    finish_3d_axes(axis, f"{name}: occupied FMM leaves (max depth {maximum_depths[name]})")
plt.tight_layout()

level_pairs = [(name, item.source_level, item.target_level)
               for name, topology in topologies.items()
               for item in topology.m2l_interactions]
if level_pairs:
    display(pd.DataFrame(level_pairs,
                         columns=["tree", "source_level", "target_level"])
            .value_counts().rename("interactions").to_frame())

## 3. Inspect actual interaction records

Same-level M2L retains the classical list2 partition. Unequal boxes use non-touching bounds plus the convergence-safe enclosing-sphere criterion $(r_s+r_t)/d\leq0.75$. Blue and cyan show same- and unequal-level P2P; orange and purple show same- and cross-level M2L. The ancestor toggle includes M2L received by ancestors and inherited through L2L. Empty categories remain visible.

In [ ]:
selected = (select_leaf(topologies["adaptive"])
            if SELECTED_LEAF is None else SELECTED_LEAF)


def plot_comparison(selected_leaf=selected, include_ancestors=INCLUDE_ANCESTORS):
    adaptive_node = topologies["adaptive"].nodes[selected_leaf]
    uniform_id = matching_node(topologies["uniform"], adaptive_node)
    uniform_node = topologies["uniform"].nodes[uniform_id]
    exact_match = (
        uniform_node.level == adaptive_node.level
        and np.isclose(uniform_node.half_width, adaptive_node.half_width)
        and np.allclose(vec3_to_array(uniform_node.centre),
                        vec3_to_array(adaptive_node.centre))
    )
    match_text = "same geometric box" if exact_match else "containing uniform box"
    fig = plt.figure(figsize=(15, 7))
    for panel, name, node_id in [(1, "adaptive", selected_leaf),
                                 (2, "uniform", uniform_id)]:
        axis = fig.add_subplot(1, 2, panel, projection="3d")
        draw_interactions(axis, topologies[name], node_id, include_ancestors)
        suffix = "selected adaptive leaf" if name == "adaptive" else match_text
        axis.set_title(
            f"{name.title()}: node {node_id}, depth "
            f"{topologies[name].nodes[node_id].level}\n{suffix}; "
            f"ancestors={include_ancestors}")
    plt.tight_layout()
    plt.show()


plot_comparison()
try:
    import ipywidgets as widgets
    choices = [leaf.node for leaf in topologies["adaptive"].target_leaves
               if topologies["adaptive"].nodes[leaf.node].level ==
               adaptive_tree.topology.maximum_level]
    widgets.interact(
        plot_comparison,
        selected_leaf=widgets.Dropdown(options=choices, value=selected),
        include_ancestors=widgets.Checkbox(value=INCLUDE_ANCESTORS),
    )
except ImportError:
    print("ipywidgets unavailable; edit SELECTED_LEAF and INCLUDE_ANCESTORS above.")

## 4. Build and benchmark the numerical plans

This is the first operator-allocation step. The current configuration builds one target-owned TensorDictionary plan for each tree. The case matrix machinery remains available for targeted executor experiments, but both sweep switches are disabled. The current static-plan precision is FP32. Every plan is built once, warmed once, reused for all seeded moment states, and released before the next case. Dictionary cases fail immediately unless the resolved packing is the Tensor6 dictionary.

The diagnostic tables include canonical leaf rectangles, effective particle interactions, same/unequal-level work, unique tensors, dictionary tokens and widths, host/device bytes, resolved P2P executor, setup stages, wall times, and phase medians. CUDA phases may overlap and therefore need not sum to wall time.

In [ ]:
backend_available = (BACKEND != cdfmm.ExecutionBackend.CUDA_FULL or
                     cdfmm.cuda_full_available())
can_benchmark = (RUN_BENCHMARK and backend_available and
                 not PLAN_BUILD_BLOCKED)
if PLAN_BUILD_BLOCKED:
    print("Numerical cells skipped by the memory preflight guard.")
elif not backend_available:
    print("Numerical cells skipped. Use a working CUDA device or explicitly select CPU_STATIC.")
elif not RUN_BENCHMARK:
    print("Numerical cells skipped because RUN_BENCHMARK is False.")
else:
    states = moment_states(material, N_EVALUATIONS, SEED + 1)
    # benchmark calls plan.evaluate_components only for REFERENCE_RUNS.
    plan_setup, measurements, fields, diagnostics, component_fields = benchmark(
        topologies, states, material, BACKEND, ORDER,
        interaction_modes=BENCHMARK_INTERACTION_MODES,
        reduced_values=REDUCED_VALUES, precision=PRECISION,
        p2p_modes=BENCHMARK_P2P_MODES,
        dictionary_executors=BENCHMARK_DICTIONARY_EXECUTORS,
        component_runs=REFERENCE_RUNS,
    )
    # plan_diagnostics obtains retained CUDA bytes from plan.cuda_plan_statistics.
    result_records = benchmark_summary(plan_setup, measurements, diagnostics)
    for row in result_records:
        row["tree_seconds"] = tree_times[row["tree"]]
        row["total_setup_seconds"] = row["tree_seconds"] + row["plan_setup_seconds"]
        row["configured_depth"] = maximum_depths[row["tree"]]
        row["case"] = p2p_case_label(row["p2p_mode"], row["dictionary_executor"])
    result_table = pd.DataFrame(result_records)
    if "phase_cuda_p2p_kernel_median_seconds" not in result_table:
        result_table["phase_cuda_p2p_kernel_median_seconds"] = (
            result_table["phase_p2p_median_seconds"])

    readable_results = result_table.assign(
        **{
            "Evaluation median (ms)": result_table.evaluation_median_seconds * 1e3,
            "Evaluation range (ms)": result_table.apply(
                lambda row: (f"{row.evaluation_min_seconds*1e3:.3f}–"
                             f"{row.evaluation_max_seconds*1e3:.3f}"), axis=1),
            "P2P kernel (ms)": result_table.phase_cuda_p2p_kernel_median_seconds * 1e3,
            "Plan setup (s)": result_table.plan_setup_seconds,
            "Host retained (GiB)": result_table.retained_bytes / 2**30,
            "Device retained (GiB)": result_table.device_bytes / 2**30,
            "Topology (MiB)": result_table.topology_bytes / 2**20,
            "P2P pairs (million)": result_table.p2p_interactions / 1e6,
            "M2L (million)": result_table.m2l_interactions / 1e6,
            "Cross-level M2L": result_table.cross_level_m2l_interactions,
        }
    )
    display(readable_results[[
        "tree", "configured_depth", "interaction_mode", "case",
        "resolved_packing", "resolved_p2p_executor",
        "Evaluation median (ms)", "Evaluation range (ms)", "P2P kernel (ms)",
        "Plan setup (s)", "Host retained (GiB)", "Device retained (GiB)",
        "Topology (MiB)", "P2P pairs (million)", "M2L (million)",
        "Cross-level M2L",
    ]].rename(columns={
        "tree": "Tree", "configured_depth": "Depth",
        "interaction_mode": "Geometry", "case": "P2P implementation",
        "resolved_packing": "Resolved packing",
        "resolved_p2p_executor": "Resolved executor",
    }).style.format({
        "Evaluation median (ms)": "{:.3f}", "P2P kernel (ms)": "{:.3f}",
        "Plan setup (s)": "{:.3f}", "Host retained (GiB)": "{:.3f}",
        "Device retained (GiB)": "{:.3f}", "Topology (MiB)": "{:.1f}",
        "P2P pairs (million)": "{:.2f}", "M2L (million)": "{:.3f}",
        "Cross-level M2L": "{:,.0f}",
    }))

    ratio_table = pd.DataFrame(comparison_ratios(result_records))
    display(ratio_table)

    comparison_rows = []
    comparison_keys = ["interaction_mode", "p2p_mode", "dictionary_executor"]
    for key, rows in result_table.groupby(comparison_keys):
        indexed = rows.set_index("tree")
        if not {"adaptive", "uniform"}.issubset(indexed.index):
            continue
        adaptive = indexed.loc["adaptive"]
        uniform = indexed.loc["uniform"]
        comparison_rows.append(dict(
            Geometry=key[0],
            **{
                "P2P implementation": p2p_case_label(key[1], key[2]),
                "Adaptive speedup": (uniform.evaluation_median_seconds /
                                     adaptive.evaluation_median_seconds),
                "Adaptive evaluation change": (
                    adaptive.evaluation_median_seconds /
                    uniform.evaluation_median_seconds - 1.0),
                "Adaptive device-memory change": (
                    adaptive.device_bytes / uniform.device_bytes - 1.0
                    if uniform.device_bytes else np.nan),
                "Adaptive P2P-pair change": (
                    adaptive.p2p_interactions / uniform.p2p_interactions - 1.0),
                "Adaptive M2L change": (
                    adaptive.m2l_interactions / uniform.m2l_interactions - 1.0),
            },
        ))
    comparison_table = pd.DataFrame(comparison_rows)
    display(comparison_table.style.format({
        "Adaptive speedup": "{:.2f}×",
        "Adaptive evaluation change": "{:+.1%}",
        "Adaptive device-memory change": "{:+.1%}",
        "Adaptive P2P-pair change": "{:+.1%}",
        "Adaptive M2L change": "{:+.1%}",
    }))

    timing_table = pd.DataFrame([
        {key: value for key, value in row.items() if key != "phases"}
        for row in measurements
    ])
    phase_table = pd.DataFrame([
        dict(tree=row["tree"], interaction_mode=row["interaction_mode"],
             p2p_mode=row["p2p_mode"],
             dictionary_executor=row["dictionary_executor"],
             reduced=row["reduced"], run=row["run"], **row["phases"])
        for row in measurements
    ])

    # Performance, memory and operator workload in one human-readable dashboard.
    case_keys = list(dict.fromkeys(
        (row.interaction_mode, row.p2p_mode, row.dictionary_executor)
        for row in result_table.itertuples()))
    case_labels = [f"{mode}\n{p2p_case_label(p2p, executor)}"
                   for mode, p2p, executor in case_keys]
    x = np.arange(len(case_keys), dtype=float)
    width = 0.34
    fig, axes = plt.subplots(2, 3, figsize=(17, 9))
    metric_specs = [
        (axes[0, 0], "evaluation_median_seconds", 1e3,
         "End-to-end evaluation", "ms"),
        (axes[0, 1], "plan_setup_seconds", 1.0,
         "Numerical-plan construction", "seconds"),
        (axes[1, 0], "p2p_interactions", 1e-6,
         "Near field: P2P work", "million particle pairs"),
        (axes[1, 1], "m2l_interactions", 1e-6,
         "Far field: M2L work", "million translations"),
        (axes[1, 2], "phase_cuda_p2p_kernel_median_seconds", 1e3,
         "CUDA P2P kernel", "ms"),
    ]
    for axis, field, scale, title, ylabel in metric_specs:
        for tree_index, tree in enumerate(tree_order):
            values = []
            for mode, p2p, executor in case_keys:
                rows = result_table[
                    (result_table.tree == tree)
                    & (result_table.interaction_mode == mode)
                    & (result_table.p2p_mode == p2p)
                    & (result_table.dictionary_executor == executor)]
                values.append(float(rows.iloc[0][field]) * scale
                              if len(rows) else np.nan)
            offset = (tree_index - 0.5) * width
            axis.bar(x + offset, values, width,
                     color=TREE_COLOURS[tree], label=tree.title())
        axis.set(title=title, ylabel=ylabel)
        axis.grid(axis="y", alpha=.25)

    memory_axis = axes[0, 2]
    memory_width = width / 2
    for tree_index, tree in enumerate(tree_order):
        host, device = [], []
        for mode, p2p, executor in case_keys:
            rows = result_table[
                (result_table.tree == tree)
                & (result_table.interaction_mode == mode)
                & (result_table.p2p_mode == p2p)
                & (result_table.dictionary_executor == executor)]
            host.append(float(rows.iloc[0].retained_bytes) / 2**30
                        if len(rows) else np.nan)
            device.append(float(rows.iloc[0].device_bytes) / 2**30
                          if len(rows) else np.nan)
        centre = x + (tree_index - 0.5) * width
        memory_axis.bar(centre - memory_width/2, host, memory_width,
                        color=TREE_COLOURS[tree], alpha=.45,
                        label=f"{tree.title()} host")
        memory_axis.bar(centre + memory_width/2, device, memory_width,
                        color=TREE_COLOURS[tree], hatch="//",
                        label=f"{tree.title()} device")
    memory_axis.set(title="Retained plan memory", ylabel="GiB")
    memory_axis.grid(axis="y", alpha=.25)
    memory_axis.legend(fontsize=8)

    for axis in axes.flat:
        axis.set_xticks(x, case_labels, rotation=15, ha="right")
    axes[0, 0].legend()
    fig.suptitle("FMM performance, memory and interaction workload", fontsize=14)
    fig.tight_layout()

    # Sweep plot: tree uses colour; P2P implementation uses marker and line style.
    interaction_modes = list(dict.fromkeys(timing_table.interaction_mode))
    fig, axes = plt.subplots(1, len(interaction_modes),
                             figsize=(7 * len(interaction_modes), 5),
                             squeeze=False)
    for axis, interaction_mode in zip(axes.flat, interaction_modes):
        selected_rows = timing_table[
            timing_table.interaction_mode == interaction_mode]
        for key, rows in selected_rows.groupby(
                ["tree", "p2p_mode", "dictionary_executor"]):
            tree, p2p_mode, dictionary_executor = key
            style = benchmark_style(tree, p2p_mode, dictionary_executor)
            axis.plot(rows.run, rows.seconds * 1e3,
                      linewidth=1.8, markersize=5, **style,
                      label=(f"{tree.title()} · "
                             f"{p2p_case_label(p2p_mode, dictionary_executor)}"))
        axis.set(title=interaction_mode, xlabel="moment state",
                 ylabel="synchronised evaluation (ms)")
        axis.grid(alpha=.25)
        axis.legend(fontsize=8)
    fig.suptitle("Repeated evaluation timing", fontsize=14)
    fig.tight_layout()

    # Detailed tables remain available below the figures for diagnosis.
    display(timing_table)
    display(phase_table)

## 5. Direct accuracy and far/P2P components

The direct reference uses all point sources and a fixed reproducible sample of targets without retaining a dense $N^2$ Tensor6 matrix. Particle identities remove singular self-pairs.

Adaptive and uniform near/far components generally differ because their partitions differ. Each P2P component is checked against direct summation over its own canonical leaf rows; its far component is checked against total direct minus that near reference. The total FMM error is always assessed against the matching physical direct model.

In [ ]:
if can_benchmark:
    reference_indices = reference_target_indices(
        len(positions), REFERENCE_TARGET_COUNT, SEED + 2)
    reference_runs = sorted({run % len(states) for run in REFERENCE_RUNS})
    use_cuda_direct = BACKEND == cdfmm.ExecutionBackend.CUDA_FULL
    references = {}
    for mode in BENCHMARK_INTERACTION_MODES:
        reference_plan = build_direct_reference(
            material, mode, reference_indices, cuda=use_cuda_direct)
        for run in reference_runs:
            references[mode, run] = evaluate_direct_reference(
                reference_plan, states[run])
        del reference_plan
        gc.collect()

    accuracy_rows, component_rows = [], []
    benchmark_cases = sorted({(row["p2p_mode"],
                              row["dictionary_executor"])
                             for row in result_records})
    for mode in BENCHMARK_INTERACTION_MODES:
        for run in reference_runs:
            reference = references[mode, run]
            reference_norm = max(np.linalg.norm(reference), 1e-30)
            for tree in topologies:
                near_reference = direct_near(
                    topologies[tree], material, states[run], mode,
                    reference_indices)
                for p2p_mode, dictionary_executor in benchmark_cases:
                    key = (tree, mode, p2p_mode, dictionary_executor)
                    reduced = p2p_mode == "dictionary"
                    actual = fields[key][run][reference_indices]
                    accuracy_rows.append(dict(
                        tree=tree, interaction_mode=mode,
                        p2p_mode=p2p_mode,
                        dictionary_executor=dictionary_executor,
                        p2p_implementation=p2p_case_label(
                            p2p_mode, dictionary_executor),
                        reduced=bool(reduced), run=run,
                        reference_norm=reference_norm,
                        **field_errors(actual, reference),
                    ))
                    sampled = {name: np.asarray(value)[reference_indices]
                               for name, value in component_fields[key, run].items()}
                    scale = max(np.linalg.norm(sampled["H_total"]), 1e-30)
                    assert np.linalg.norm(sampled["H_far"] + sampled["H_p2p"] -
                                          sampled["H_total"]) <= 2e-6 * scale
                    for component, truth in [
                        ("H_p2p", near_reference),
                        ("H_far", reference - near_reference),
                    ]:
                        component_rows.append(dict(
                            tree=tree, interaction_mode=mode,
                            p2p_mode=p2p_mode,
                            dictionary_executor=dictionary_executor,
                            p2p_implementation=p2p_case_label(
                                p2p_mode, dictionary_executor),
                            reduced=bool(reduced), run=run, component=component,
                            field_norm=np.linalg.norm(sampled[component]),
                            **field_errors(sampled[component], truth),
                        ))
    accuracy_table = pd.DataFrame(accuracy_rows)
    component_table = pd.DataFrame(component_rows)

    accuracy_summary = accuracy_table.groupby([
        "tree", "interaction_mode", "p2p_implementation"
    ], as_index=False).agg(
        relative_l2=("relative_l2", "median"),
        absolute_rms=("absolute_rms", "median"),
        absolute_max=("absolute_max", "max"),
    )
    display(accuracy_summary.rename(columns={
        "tree": "Tree", "interaction_mode": "Geometry",
        "p2p_implementation": "P2P implementation",
        "relative_l2": "Median relative L2 error",
        "absolute_rms": "Median absolute RMS error",
        "absolute_max": "Maximum absolute error",
    }).style.format({
        "Median relative L2 error": "{:.3e}",
        "Median absolute RMS error": "{:.4g}",
        "Maximum absolute error": "{:.4g}",
    }))

    # Accuracy dashboard. Each geometry mode gets its own panel set during a sweep.
    for mode in BENCHMARK_INTERACTION_MODES:
        fig, axes = plt.subplots(2, 2, figsize=(13, 9))
        mode_accuracy = accuracy_table[accuracy_table.interaction_mode == mode]
        mode_components = component_table[component_table.interaction_mode == mode]
        for key, rows in mode_accuracy.groupby(
                ["tree", "p2p_mode", "dictionary_executor"]):
            tree, p2p_mode, dictionary_executor = key
            style = benchmark_style(tree, p2p_mode, dictionary_executor)
            label = f"{tree.title()} · {p2p_case_label(p2p_mode, dictionary_executor)}"
            axes[0, 0].plot(rows.run, rows.relative_l2, linewidth=1.8,
                            markersize=6, label=label, **style)
            axes[0, 1].plot(rows.run, rows.absolute_rms, linewidth=1.8,
                            markersize=6, label=label, **style)
        for component, axis in [("H_p2p", axes[1, 0]),
                                ("H_far", axes[1, 1])]:
            selected_components = mode_components[
                mode_components.component == component]
            for key, rows in selected_components.groupby(
                    ["tree", "p2p_mode", "dictionary_executor"]):
                tree, p2p_mode, dictionary_executor = key
                style = benchmark_style(tree, p2p_mode, dictionary_executor)
                label = (f"{tree.title()} · "
                         f"{p2p_case_label(p2p_mode, dictionary_executor)}")
                axis.plot(rows.run, rows.relative_l2, linewidth=1.8,
                          markersize=6, label=label, **style)
        axes[0, 0].set(title="Total field accuracy", ylabel="relative L2 error",
                       yscale="log")
        axes[0, 1].set(title="Total field absolute error", ylabel="RMS field error")
        axes[1, 0].set(title="Near-field component accuracy",
                       ylabel="P2P relative L2 error", yscale="log")
        axes[1, 1].set(title="Far-field component accuracy",
                       ylabel="far-field relative L2 error", yscale="log")
        for axis in axes.flat:
            axis.set_xlabel("reference moment state")
            axis.grid(alpha=.25)
        axes[0, 0].legend(fontsize=8)
        fig.suptitle(f"Accuracy against direct evaluation: {mode}", fontsize=14)
        fig.tight_layout()

    difference_rows = []
    for mode in BENCHMARK_INTERACTION_MODES:
        for run in reference_runs:
            reference_norm = max(np.linalg.norm(references[mode, run]), 1e-30)
            for p2p_mode, dictionary_executor in benchmark_cases:
                reduced = p2p_mode == "dictionary"
                difference_norm = np.linalg.norm(
                    fields["adaptive", mode, p2p_mode,
                           dictionary_executor][run][reference_indices]
                    - fields["uniform", mode, p2p_mode,
                             dictionary_executor][run][reference_indices])
                difference_rows.append(dict(
                    comparison="adaptive - uniform", interaction_mode=mode,
                    p2p_mode=p2p_mode,
                    dictionary_executor=dictionary_executor,
                    p2p_implementation=p2p_case_label(
                        p2p_mode, dictionary_executor),
                    reduced=bool(reduced), run=run,
                    difference_norm=difference_norm,
                    relative_difference=difference_norm / reference_norm,
                ))
            if (("canonical", "source_warp") in benchmark_cases and
                    ("dictionary", "source_warp") in benchmark_cases):
                for tree in topologies:
                    difference_norm = np.linalg.norm(
                        fields[tree, mode, "dictionary",
                               "source_warp"][run][reference_indices]
                        - fields[tree, mode, "canonical",
                                 "source_warp"][run][reference_indices])
                    difference_rows.append(dict(
                        comparison="reduced - ordinary", tree=tree,
                        interaction_mode=mode, p2p_mode="dictionary",
                        dictionary_executor="source_warp",
                        p2p_implementation="Reduced source-warp",
                        reduced=True, run=run,
                        difference_norm=difference_norm,
                        relative_difference=difference_norm / reference_norm,
                    ))
    difference_table = pd.DataFrame(difference_rows)
    display(difference_table.style.format({
        "difference_norm": "{:.4g}", "relative_difference": "{:.3e}"}))

    # Detailed component rows remain available for numerical diagnosis.
    display(component_table)

## Reading the results

- **Colours are fixed by tree:** uniform is blue and adaptive is red.
- **Markers and line styles identify the P2P implementation:** canonical, BSR, reduced source-warp, reduced target-owned, or reduced power2. Geometry modes are shown in separate panels during a full sweep.
- The headline comparison table reports adaptive speed as `uniform time / adaptive time`; values above one mean adaptive is faster. Signed percentage columns report adaptive relative to uniform.
- Near-field work is the number of explicit particle pairs. Far-field work is the number of M2L translations. CUDA phase timings overlap and must not be added to obtain wall time.
- Host retained memory, device retained memory, and the topology footprint are reported separately. The preflight host peak is only an estimate of temporary construction storage.
- Adaptive and uniform plans always use identical positions and moment states, but different configured depths intentionally compare different work/accuracy trade-offs.
- Near/far components can differ between trees because their interaction partitions differ. Total-field error against the direct reference determines accuracy.
- Changing moments reuses a plan. Changing positions, topology, depth, geometry mode, or packing rebuilds it.